# 02 — Trening modeli (szybki przykład na próbce)

Pełne treningi: `python scripts/03_train.py --model all` (każdy run → `runs/<run>/`:
model, predykcje val/test, metryki). Tu: jeden model na próbce, obie reguły decyzyjne
(argmax i kosztowa), a następnie łączenie zapisanych runów.

Klasy: indeks 0–4 = ESI 1–5.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data import early_stopping_split, ensure_prepared, get_xy, load_splits
from src.data.dataset import sample_frame
from src.evaluation.metrics import decide, full_evaluation
from src.evaluation.visualizations import plot_confusion_matrix
from src.models import MODEL_REGISTRY, get_model

FEATURE_SET = 'triage'
N_TRAIN, N_VAL = 20_000, 5_000       # próbki (None = cały zbiór)
KEY_METRICS = ['quadratic_weighted_kappa', 'undertriage_rate', 'overtriage_rate',
               'critical_miss_rate', 'f1_macro', 'auc_macro', 'expected_cost']

ensure_prepared()
splits = load_splits(('train', 'val'), feature_set=FEATURE_SET)
X_train, y_train = get_xy(sample_frame(splits['train'], N_TRAIN), FEATURE_SET)
X_val, y_val = get_xy(sample_frame(splits['val'], N_VAL), FEATURE_SET)
X_fit, y_fit, X_es, y_es = early_stopping_split(X_train, y_train)
print(f'fit {X_fit.shape} | early stopping {X_es.shape} | val {X_val.shape}')
print('Modele:', list(MODEL_REGISTRY))

## Szybki model (XGBoost)

`X_val`/`y_val` w `fit()` to WYŁĄCZNIE zbiór early stopping wydzielony z train —
zbiór `val` zostaje nietknięty (porównanie i łączenie modeli).

In [ ]:
model = get_model('xgboost', params={'n_estimators': 500})
model.fit(X_fit, y_fit, X_val=X_es, y_val=y_es, feature_set=FEATURE_SET)
proba = model.predict_proba(X_val)

## Reguły decyzyjne: argmax vs kosztowa

Reguła kosztowa wybiera poziom minimalizujący oczekiwany koszt błędu
(niedotriażowanie 3× droższe niż przetriażowanie).

In [ ]:
results = {rule: full_evaluation(y_val, decide(proba, rule), proba, print_report=False)
           for rule in ('argmax', 'cost')}
pd.DataFrame(results).loc[KEY_METRICS].astype(float).round(4)

In [ ]:
plot_confusion_matrix(y_val, decide(proba, 'cost'), title='Macierz pomyłek (reguła kosztowa)', show=True)

## Pełny run z artefaktami (opcjonalnie)

To samo co `python scripts/03_train.py --model lightgbm --sample 20000 --eval-sample 5000 --tag nb`.

In [ ]:
RUN_TRAINING = False   # True = zapisz run do runs/
if RUN_TRAINING:
    from src.models.train import run_training

    result = run_training('lightgbm', feature_set=FEATURE_SET,
                          train_sample=N_TRAIN, eval_sample=N_VAL, tag='nb')
    print(result['run_dir'])

## Łączenie zapisanych runów

Wagi i kalibrację (temperature scaling) dobieramy na `val`, raportujemy na `test`
dokładnie tą samą recepturą. Łączyć można tylko runy ocenione na tych samych wierszach.

In [ ]:
from src.ensemble import (aligned_probas, apply_temperature, blend, diversity, evaluate,
                          fit_temperature, latest_runs, list_runs, optimize_weights)

list_runs().tail(10)

In [ ]:
runs = latest_runs(FEATURE_SET, include_samples=True)   # na pełnych danych: include_samples=False
same_rows = [r for r in runs if r.info.get('eval_sample') == runs[0].info.get('eval_sample')]
print([r.name for r in same_rows])

y_v, P_v = aligned_probas(same_rows, 'val')
weights = optimize_weights(P_v, y_v)
T = fit_temperature(blend(P_v, weights), y_v)
print('wagi:', weights, '| T =', round(T, 3))
diversity(P_v, y_v).round(3)

In [ ]:
y_t, P_t = aligned_probas(same_rows, 'test')
p_t = apply_temperature(blend(P_t, weights), T)
pd.DataFrame({rule: evaluate(y_t, p_t, rule) for rule in ('argmax', 'cost')}).loc[KEY_METRICS].astype(float).round(4)